In [3]:
import os
import random
import requests
import numpy as np
import pandas as pd
from PIL import Image
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import cv2
from collections import Counter

def get_random_images(filepath, n=10):
    """Select n random images from the specified directory."""
    all_images = [f for f in os.listdir(filepath) if f.startswith('tt')]
    return random.sample(all_images, min(n, len(all_images)))

def extract_imdb_id(filename):
    """Extract IMDB ID from filename."""
    return filename.split('.')[0]

def get_imdb_data(imdb_id):
    """Fetch and parse IMDB page data."""
    url = f'https://www.imdb.com/title/{imdb_id}/'
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        script = soup.find('script', {'type': 'application/ld+json'})
        if not script:
            return None
        
        import json
        data = json.loads(script.string)
        
        show_data = {
            'imdb_id': imdb_id,
            'title': data.get('name', ''),
            'rating': data.get('aggregateRating', {}).get('ratingValue', 0),
            'rating_count': data.get('aggregateRating', {}).get('ratingCount', 0),
            'content_rating': data.get('contentRating', ''),
            'date_published': data.get('datePublished', ''),
            'genre_count': len(data.get('genre', [])),
            'actor_count': len(data.get('actor', []))
        }
        
        return show_data
    except Exception as e:
        print(f"Error processing {imdb_id}: {str(e)}")
        return None

def analyze_image(image_path):
    """Perform computer vision analysis on an image."""
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    # Convert to RGB for better color analysis
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize for consistency
    img_resized = cv2.resize(img_rgb, (224, 336))
    
    # 1. Color Analysis
    # Calculate average color
    avg_color = np.mean(img_resized, axis=(0, 1))
    
    # 2. Calculate color histogram
    hist_features = []
    for i in range(3):
        hist = cv2.calcHist([img_resized], [i], None, [8], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        hist_features.extend(hist)
    
    # 3. Edge Detection
    gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.mean(edges > 0)
    
    # 4. Brightness and Contrast
    brightness = np.mean(gray)
    contrast = np.std(gray)
    
    # 5. Face Detection
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    face_count = len(faces)
    
    return {
        'avg_red': avg_color[0],
        'avg_green': avg_color[1],
        'avg_blue': avg_color[2],
        'edge_density': edge_density,
        'brightness': brightness,
        'contrast': contrast,
        'face_count': face_count,
        **{f'hist_feature_{i}': v for i, v in enumerate(hist_features)}
    }

def create_dataset(image_folder):
    """Create dataset from selected images and their analysis."""
    images = get_random_images(image_folder)
    data = []
    
    for img in images:
        imdb_id = extract_imdb_id(img)
        show_data = get_imdb_data(imdb_id)
        
        if show_data:
            # Analyze image
            img_path = os.path.join(image_folder, img)
            img_features = analyze_image(img_path)
            
            if img_features:
                combined_data = {**show_data, **img_features}
                data.append(combined_data)
    
    df = pd.DataFrame(data)
    df.to_csv('tv_series_analysis.csv', index=False)
    return df

def perform_pca(df):
    """Perform PCA on numerical features."""
    # Select numerical columns (excluding imdb_id and title)
    numerical_cols = df.select_dtypes(include=[np.number]).columns
    
    # Standardize the features
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df[numerical_cols])
    
    # Apply PCA
    pca = PCA()
    pca_result = pca.fit_transform(scaled_features)
    
    return pca, pca_result, numerical_cols

def create_visualizations(df, pca, pca_result, numerical_cols):
    """Create various visualizations for the analysis."""
    # Create output directory if it doesn't exist
    os.makedirs('visualizations', exist_ok=True)
    
    # 1. Scree plot
    plt.figure(figsize=(10, 6))
    explained_variance_ratio = pca.explained_variance_ratio_
    plt.plot(range(1, len(explained_variance_ratio) + 1), 
            np.cumsum(explained_variance_ratio), 'bo-')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance Ratio')
    plt.title('PCA Scree Plot')
    plt.savefig('visualizations/scree_plot.png')
    plt.close()
    
    # 2. PCA biplot
    plt.figure(figsize=(12, 8))
    plt.scatter(pca_result[:, 0], pca_result[:, 1])
    
    # Add title labels
    for i, title in enumerate(df['title']):
        plt.annotate(title, (pca_result[i, 0], pca_result[i, 1]))
    
    plt.xlabel('First Principal Component')
    plt.ylabel('Second Principal Component')
    plt.title('PCA Biplot of TV Series')
    plt.savefig('visualizations/pca_biplot.png')
    plt.close()
    
    # 3. Feature correlations with ratings
    plt.figure(figsize=(12, 6))
    correlations = df[numerical_cols].corrwith(df['rating']).sort_values()
    correlations.plot(kind='bar')
    plt.title('Feature Correlations with Rating')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('visualizations/rating_correlations.png')
    plt.close()
    
    # 4. Color analysis visualization
    plt.figure(figsize=(10, 6))
    color_data = df[['avg_red', 'avg_green', 'avg_blue']].mean()
    color_data.plot(kind='bar', color=['red', 'green', 'blue'])
    plt.title('Average Color Distribution')
    plt.savefig('visualizations/color_analysis.png')
    plt.close()

def main(image_folder):
    """Main function to run the analysis."""
    print("Creating dataset...")
    df = create_dataset(image_folder)
    
    print("\nPerforming PCA...")
    pca, pca_result, numerical_cols = perform_pca(df)
    
    print("\nCreating visualizations...")
    create_visualizations(df, pca, pca_result, numerical_cols)
    
    print("\nAnalysis complete! Check the following files:")
    print("- tv_series_analysis.csv: Raw data with image features")
    print("- visualizations/scree_plot.png: PCA explained variance")
    print("- visualizations/pca_biplot.png: TV series clustering")
    print("- visualizations/rating_correlations.png: Feature correlations")
    print("- visualizations/color_analysis.png: Color distribution")

if __name__ == "__main__":
    # Replace with your image folder path
    IMAGE_FOLDER = "tv_posters"
    main(IMAGE_FOLDER)

Creating dataset...

Performing PCA...

Creating visualizations...


c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Analysis complete! Check the following files:
- tv_series_analysis.csv: Raw data with image features
- visualizations/scree_plot.png: PCA explained variance
- visualizations/pca_biplot.png: TV series clustering
- visualizations/rating_correlations.png: Feature correlations
- visualizations/color_analysis.png: Color distribution
